# شطر | من قراءة الشعر إلى كتابته
هذا دفتر تنفيذ للمشروع الجديد. اقرئي README_AR.md أولًا. الفئات: الجاهلي والأندلسي والعثماني والحديث.

اختاري T4 GPU. لا تشغلي الاختبار النهائي قبل تثبيت القرارات. نتائج 61.20% السابقة تخص تقسيمًا مختلفًا ولا تنتمي تلقائيًا لهذه النسخة.

In [ ]:
from pathlib import Path
import os, sys, zipfile, shutil, subprocess
from google.colab import drive, files
drive.mount('/content/drive')
PACKAGE=Path('/content/Shatr_Project')
if not PACKAGE.exists():
    print('ارفعي Shatr_Project.zip من الحزمة التي نزلتيها.')
    uploaded=files.upload()
    name=next((n for n in uploaded if n.lower().endswith('.zip')),None)
    if name is None: raise ValueError('ارفعي ملف ZIP للمشروع.')
    with zipfile.ZipFile(name) as z:
        for item in z.namelist():
            if not (Path('/content')/item).resolve().is_relative_to(Path('/content')):
                raise ValueError('مسار غير آمن داخل الأرشيف')
        z.extractall('/content')
assert (PACKAGE/'app.py').exists(), PACKAGE
ROOT=Path('/content/shatr_project_v1')
PERSIST=Path('/content/drive/MyDrive/shatr_project_v1')
OLD_ROOT=Path('/content/drive/MyDrive/basma_project')
os.environ['SHATR_ROOT']=str(ROOT)
print('التشغيل المؤقت:',ROOT)
print('البيانات والتقارير الصغيرة:',PERSIST)


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PACKAGE/'requirements.txt')],check=True)


## ١. التحقق من الأكواد
هذه اختبارات مصطنعة، لا تنتج نسبة أداء للمشروع ولا تنزل نماذج.

In [ ]:
def run(*args):
    subprocess.run([sys.executable,*args],cwd=PACKAGE,check=True,env=os.environ.copy())
run('test_project.py')
run('test_features.py')


## ٢. تجهيز البيانات مرة واحدة
نستخدم الأزواج السابقة التي راجعتما ترتيب أشطرها، ثم نقسم الشعراء داخل كل عصر ببذرة ثابتة. راجعا SOURCE_REVIEW_AR.md؛ مراجعة التسميات والحقوق والأسماء البديلة لا يحسمها اختبار آلي.

لا تحذفي تقسيمًا موجودًا لإعادة المحاولة بناءً على النتيجة.

In [ ]:
if (PERSIST/'data/labels.json').exists() and not (ROOT/'data/labels.json').exists():
    shutil.copytree(PERSIST/'data',ROOT/'data')
    if (PERSIST/'outputs').exists():
        shutil.copytree(PERSIST/'outputs',ROOT/'outputs',dirs_exist_ok=True)
    print('استعدنا التقسيم المحفوظ؛ لم نعد إنشاءه.')
elif not (ROOT/'data/labels.json').exists():
    run('prepare_project.py','--old-root',str(OLD_ROOT),'--new-root',str(ROOT))
else:
    print('التقسيم موجود؛ نستخدمه كما هو.')
def sync():
    run('sync_results.py','--root',str(ROOT),'--dest',str(PERSIST))
sync()
import pandas as pd
display(pd.read_csv(ROOT/'data/data_summary.csv'))


## ٣. خط الأساس
TF-IDF يحول أنماط الحروف إلى أرقام، وLogistic Regression يتعلم تصنيفها. نقارنه بالنموذج المتخصص على نفس البيانات.

In [ ]:
run('train_classifier.py','baseline')
sync()


## ٤. تخصيص AraBERT وAraPoemBERT
ثلاث دورات كبداية، مع اختيار أفضل Macro-F1 على validation. عدلي الإعدادات في إصدار تجربة مستقل وسجليها، ولا تغيري test.

In [ ]:
run('train_classifier.py','train','--model','arabert','--epochs','3')
run('train_classifier.py','train','--model','arapoembert','--epochs','3')
run('select_classifier.py')
run('make_report.py')
sync()


## ٥. حفظ المصنف قبل إنهاء الجلسة
الأوزان كبيرة وتبقى في /content لتجنب امتلاء Drive. نزلي الأرشيف واحفظيه مع اسم التجربة. التقارير وحدها لا تشغل النموذج في جلسة جديدة.

In [ ]:
def download_folder(folder, filename):
    archive=Path('/content')/filename
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_STORED) as z:
        for path in folder.iterdir():
            if path.is_file(): z.write(path,arcname=path.name)
    files.download(str(archive))
import json
selection=json.loads((ROOT/'outputs/selected_classifier.json').read_text())
archive_name='Shatr_AraPoemBERT_Model.zip' if selection['model_key']=='arapoembert' else 'Shatr_AraBERT_Model.zip'
download_folder(ROOT/'outputs'/selection['folder'],archive_name)


## ٦. Qwen قبل التدريب
نسجل تكملات حقيقية من النموذج الأساسي؛ لا نفترض أنها جيدة قبل قراءتها.

In [ ]:
run('train_completion.py','before')
sync()


## ٧. تدريب LoRA
ندرب إضافات r=8 على q_proj وv_proj. لا نحسب خسارة على التعليمات أو الشطر الأول أو الحشو. تُحفظ نسخة الإعدادات وتوقيع البيانات، وتمنع الحماية الاستئناف بإعدادات مختلفة داخل المجلد نفسه.

In [ ]:
run('train_completion.py','train','--epochs','2')
sync()


## ٨. مقارنة قبل/بعد والتقييم البشري
المدخلات والبذور وإعدادات التوليد متطابقة. ملف A/B أعمى؛ لا ترسلي ملف المفتاح إلى المقيم. اتبعي HUMAN_EVALUATION_AR.md.

In [ ]:
run('train_completion.py','compare')
run('make_report.py')
sync()


## استعادة المصنف في جلسة جديدة — اختيارية
اتركيها False عند التدريب لأول مرة.

In [ ]:
RESTORE_CLASSIFIER = False  # True فقط لاستعادة أرشيفك في جلسة جديدة
import json
selection_file=ROOT/'outputs/selected_classifier.json'
restore_key=json.loads(selection_file.read_text())['model_key'] if selection_file.exists() else 'arabert'
restore_folder='arapoembert_model' if restore_key=='arapoembert' else 'basma_model'
if RESTORE_CLASSIFIER and not (ROOT/'outputs'/restore_folder/'model.safetensors').exists():
    print('ارفعي أرشيف المصنف المختار، من بيانات هذه النسخة نفسها.')
    uploaded=files.upload()
    filename=next((n for n in uploaded if n.lower().endswith('.zip')),None)
    if filename is None: raise ValueError('ارفعي أرشيف المصنف')
    run('restore_model.py','--archive',str(Path(filename).resolve()),'--root',str(ROOT),'--model',restore_key)


## ٩. الواجهة
إذا بعض النماذج لم تدرب بعد، افتحي الواجهة لمراجعة التصميم والبحث، وستظل ميزاتها غير المتاحة معطلة بوضوح. الواجهة عبر وكيل Colab وليست نشرًا عامًا دائمًا.

In [ ]:
import importlib.util
sys.path.insert(0,str(PACKAGE))
from app import build_app
from google.colab import output
if 'demo' in globals(): demo.close()
demo=build_app(ROOT)
demo.launch(share=False,server_name='0.0.0.0',server_port=7862,prevent_thread_lock=True)
print(output.eval_js('google.colab.kernel.proxyPort(7862)'))


## ١٠. الاختبار النهائي — يدويًا فقط
بعد تثبيت القرارات والتقييم البشري، شغلي الأمرين التاليين في خلية جديدة:

```python
run('train_classifier.py','test','--model','arabert')
run('train_classifier.py','test','--model','arapoembert')
run('train_completion.py','test')
run('make_report.py')
sync()
```

هذه فقرة نصية عمدًا حتى لا يؤدي Run all إلى تقييم test دون قصد.

## استعادة المصنف في جلسة جديدة
بعد الإعداد واستعادة البيانات، ارفعي أرشيف المصنف المختار واستعيديه بالخلية الاختيارية أعلاه. لا تنقلي أوزان تجربة أخرى. التطبيق يفحص توقيع البيانات قبل الاستخدام. ملف shatr_adapter يحتاج أوزان Qwen الأساسية، ويمكن تنزيلها عند أول طلب تكملة.